# Orchestrating Jobs, Model Registration, and Continuous Deployment with Amazon SageMaker

**This repository:** MTI (Maritime Transparency Index) score prediction — preprocessing, LightGBM training, evaluation, and conditional model registration. The pipeline module is `pipelines.mti_score.pipeline`. Pushes to your connected branch run **CodeBuild** (`codebuild-buildspec.yml`), which executes `run-pipeline` for the same module; this notebook is for interactive development in SageMaker Studio.

Amazon SageMaker offers Machine Learning application developers and Machine Learning operations engineers the ability to orchestrate SageMaker jobs and author reproducible Machine Learning pipelines, deploy custom-build models for inference in real-time with low latency or offline inferences with Batch Transform, and track lineage of artifacts. You can institute sound operational practices in deploying and monitoring production workflows, deployment of model artifacts, and track artifact lineage through a simple interface, adhering to safety and best-practice paradigms for Machine Learning application development.

The SageMaker Pipelines service supports a SageMaker Machine Learning Pipeline Domain Specific Language (DSL), which is a declarative JSON specification. This DSL defines a Directed Acyclic Graph (DAG) of pipeline parameters and SageMaker job steps. The SageMaker Python SDK streamlines the generation of the pipeline DSL using constructs that are already familiar to engineers and scientists alike.

The SageMaker Model Registry is where trained models are stored, versioned, and managed. Data Scientists and Machine Learning Engineers can compare model versions, approve models for deployment, and deploy models from different AWS accounts, all from a single Model Registry.

## SageMaker Pipelines

Amazon SageMaker Pipelines support the following activites:

* Pipelines - A Directed Acyclic Graph of steps and conditions to orchestrate SageMaker jobs and resource creation.
* Processing Job steps - A simplified, managed experience on SageMaker to run data processing workloads, such as feature engineering, data validation, model evaluation, and model interpretation.
* Training Job steps - An iterative process that teaches a model to make predictions by presenting examples from a training dataset.
* Conditional step execution - Provides conditional execution of branches in a pipeline.
* Registering Models - Creates a model package resource in the Model Registry that can be used to create deployable models in Amazon SageMaker.
* Creating Model steps - Create a model for use in transform steps or later publication as an endpoint.
* Parameterized Pipeline executions - Allows pipeline executions to vary by supplied parameters.
* Transform Job steps - A batch transform to preprocess datasets to remove noise or bias that interferes with training or inference from your dataset, get inferences from large datasets, and run inference when you don't need a persistent endpoint.

## Layout of this project (MTI score prediction)

The SageMaker ModelBuild template layout is adapted for **MTI score forecasting** (feature engineering in `preprocess.py`, LightGBM in `train.py`, metrics in `evaluate.py`). CI/CD runs `codebuild-buildspec.yml`, which calls `run-pipeline --module-name pipelines.mti_score.pipeline`.

```
|-- codebuild-buildspec.yml
|-- MANIFEST.in
|-- CONTRIBUTING.md
|-- pipelines
|   |-- mti_score
|   |   |-- evaluate.py
|   |   |-- __init__.py
|   |   |-- pipeline.py
|   |   |-- preprocess.py
|   |   |-- train.py
|   |   `-- requirements.txt
|   |-- get_pipeline_definition.py
|   |-- __init__.py
|   |-- run_pipeline.py
|   |-- _utils.py
|   `-- __version__.py
|-- README.md
|-- sagemaker-pipelines-project.ipynb
|-- setup.cfg
|-- setup.py
|-- tests
|   `-- test_pipelines.py
`-- tox.ini
```

A description of some of the artifacts is provided below.

**CodeBuild execution:** `codebuild-buildspec.yml` installs this package and runs `run-pipeline` (same `get_pipeline` module as in this notebook).

**Pipeline code (`pipelines/mti_score/`):**

- `pipeline.py` — defines `get_pipeline()` (preprocess → train → evaluate → conditional register).
- `preprocess.py` — downloads CSV from S3 URI `InputDataUrl`, builds lags and writes `prepared.csv`.
- `train.py` — LightGBM multi-horizon training; writes `predictions.csv` into the model artifact.
- `evaluate.py` — reads predictions, writes `evaluation.json` (MSE for the condition step).
- `requirements.txt` — **LightGBM only** (pinned); shipped beside `train.py` in `source_dir` for `pip install -r`. Do not pass it again as `SKLearn(dependencies=[...])` (duplicates the file in the tarball). Packaged via `setup.py` / `MANIFEST.in`.

**Packaging non-Python files:** `requirements.txt` is listed in `package_data` / `MANIFEST.in` so `pip install .` in CodeBuild includes it under `site-packages/pipelines/mti_score/`.

Example `setup.py` fragment:

```
    packages=setuptools.find_packages(),
    include_package_data=True,
    package_data={
        "pipelines.mti_score": ["requirements.txt"],
    },
```

**CLI utilities:** `get_pipeline_definition.py`, `run_pipeline.py`, `_utils.py`, `__version__.py`.

**Tests:** `tests/test_pipelines.py` · **tooling:** `setup.cfg`, `tox.ini`.

### A SageMaker Pipeline

The MTI pipeline follows: **preprocessing** (SKLearnProcessor) → **training** (LightGBM via SKLearn script mode) → **evaluation** (ScriptProcessor) → **conditional model registration** if MSE is within the threshold.

![A typical ML Application pipeline](img/pipeline-full.png)

### Getting some constants

We read Region, execution role, and default bucket from the Studio / notebook environment.

In [ ]:
import boto3
import sagemaker


region = boto3.Session().region_name
role = sagemaker.get_execution_role()
default_bucket = sagemaker.session.Session().default_bucket()

# Optional: rename model package group and pipeline for experiments in Studio (CodeBuild may override via run-pipeline kwargs)
model_package_group_name = "MTIScoreModelPackageGroup-Example"
pipeline_name = "MTIScorePipeline-Example"

### Get the pipeline instance

Here we get the pipeline instance from your pipeline module so that we can work with it.

In [ ]:
from pipelines.mti_score.pipeline import get_pipeline


pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=default_bucket,
    model_package_group_name=model_package_group_name,
    pipeline_name=pipeline_name,
)

### Submit the pipeline to SageMaker and start execution

Let's submit our pipeline definition to the workflow service. The role passed in will be used by the workflow service to create all the jobs defined in the steps.

In [ ]:
pipeline.upsert(role_arn=role)

We'll start the pipeline with default parameters. You can override pipeline parameters when calling `start()` (see below). Current parameters include `InputDataUrl` (S3 URI to your MTI CSV), `ProcessingInstanceCount`, `MseThreshold`, and `ModelApprovalStatus`. 

In [ ]:
execution = pipeline.start()

### Pipeline Operations: examining and waiting for pipeline execution

Now we describe execution instance and list the steps in the execution to find out more about the execution.

In [ ]:
execution.describe()

We can wait for the execution by invoking `wait()` on the execution:

In [ ]:
execution.wait()

We can list the execution steps to check out the status and artifacts:

In [ ]:
execution.list_steps()

### Parameterized executions

Pass a `parameters` dict to `pipeline.start()` to override defaults. Names must match the pipeline definition: `InputDataUrl`, `ProcessingInstanceCount`, `MseThreshold`, `ModelApprovalStatus`.

Instance types for processing and training are fixed in `pipeline.py` (they cannot be pipeline variables in this SDK because image URIs are resolved at definition time). To use a different instance size, change the defaults passed into `get_pipeline(..., processing_instance_type=..., training_instance_type=...)` in code, or call `get_pipeline` with those kwargs from this notebook.

Setting `ModelApprovalStatus` to `Approved` means a passing model package can proceed to deployment pipelines that require approval.

In [ ]:
execution = pipeline.start(
    parameters=dict(
        InputDataUrl=f"s3://sagemaker-servicecatalog-seedcode-{region}/dataset/mti_scores_history.csv",
        ProcessingInstanceCount=1,
        MseThreshold=0.05,
        ModelApprovalStatus="Approved",
    )
)

In [ ]:
execution.wait()

In [ ]:
execution.list_steps()